# MTP rejection：长尾、频率与 confidence gate 分析

本 notebook 使用完整 trace 统计 pair 频率，使用随机抽样并经 V4 judge 的事件分析误拒候选与各信号的关系。

- `supported_candidate = PROVEN_VALID + PLAUSIBLE_UNPROVEN`：有正面证据的误拒候选；
- `INSUFFICIENT` 单独报告，不直接算成误拒；
- `not_proven_invalid` 仅作为上界敏感性分析，不能称为误拒率；
- frequency 来自完整 trace；ratio、entropy 等来自被 judge 的事件。

In [ ]:
from pathlib import Path
from collections import Counter
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

REPO = Path('/root/sglang-dspark-csd')
TRACE_ROOT = REPO / 'runs/mtp_csd/qwen35b_mtp314_rejection_trace/runs/trace_large_80_30_80_20260814_001505'
JUDGE_ROOT = REPO / 'runs/mtp_csd/qwen35b_mtp314_rejection_trace/runs/large_v4_sample3_20260814_015007'
CALIBRATION = REPO / 'runs/mtp_csd/qwen35b_mtp314_final/calibration/csd_table_redpajama_logits_ungated_6domains_n1000_Qwen3.5-35B-A3B_mtp_EAGLE_steps3_topk1_draft3_temp1.0_ratio0.01.json'
MODEL = Path('/data/model/Qwen3.5-35B-A3B')
TASKS = {
    'LCB': ('lcb_v6', 'lcb_v6'),
    'AIME': ('aime25', 'aime25'),
    'OlympiadBench': ('olympiad_math_en', 'olympiad_math_en'),
}
for p in [TRACE_ROOT, JUDGE_ROOT, CALIBRATION, MODEL]:
    assert p.exists(), p
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
def token_pair(x):
    return int(x['draft_token_id']), int(x['residual_token_id'])

# 完整 trace：定义每个 pair 的实际出现频率。
task_frequency = {}
for task, (trace_dir, _) in TASKS.items():
    counter = Counter()
    path = TRACE_ROOT / trace_dir / 'trace/events.rank0.jsonl'
    with path.open() as f:
        for line in f:
            if line.strip():
                counter[token_pair(json.loads(line))] += 1
    task_frequency[task] = counter
overall_frequency = sum(task_frequency.values(), Counter())

# Calibration frequency。GPU active table 的正式阈值是 6。
calibration_obj = json.loads(CALIBRATION.read_text())
calibration_frequency = {
    (int(e['lhs_token']), int(e['rhs_token'])): int(e['freq'])
    for e in calibration_obj['entries']
}

# Judge 样本：加入完整 trace frequency 和标签派生字段。
records = []
for task, (_, judge_dir) in TASKS.items():
    path = JUDGE_ROOT / judge_dir / 'judge/judged_v4.jsonl'
    with path.open() as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            src = obj['source_event']
            pair = token_pair(src)
            label = obj['mapped_judgment']['draft_validity']
            row = dict(src)
            row.update({
                'task': task, 'pair': pair, 'label': label,
                'trace_frequency_task': task_frequency[task][pair],
                'trace_frequency_overall': overall_frequency[pair],
                'calibration_frequency': calibration_frequency.get(pair, 0),
                'supported_candidate': label in {'PROVEN_VALID', 'PLAUSIBLE_UNPROVEN'},
                'insufficient': label == 'INSUFFICIENT',
                'proven_invalid': label == 'PROVEN_INVALID',
                'not_proven_invalid': label != 'PROVEN_INVALID',
            })
            records.append(row)
df = pd.DataFrame(records)
print(f'完整 trace: {sum(overall_frequency.values()):,} events, {len(overall_frequency):,} unique pairs')
print(f'Judge 样本: {len(df):,}')
display(df['label'].value_counts().to_frame('count').assign(rate=lambda x: x['count']/len(df)))

## 1.1 固定全体拒绝的头部，测量其对误拒候选的覆盖

这里不能在误拒候选内部重新排序。先用完整 trace 固定 Top-K% 高频 pair，再计算这些同一批 pair 覆盖多少 Judge 误拒候选；由此区分覆盖能力与判别能力。

In [ ]:
def fixed_head_coverage(counter, judged_df):
    order = sorted(counter, key=counter.get, reverse=True)
    total_events = sum(counter.values())
    candidate_df = judged_df[judged_df['supported_candidate']]
    rows = []
    for q in [0.01, 0.05, 0.10, 0.20]:
        k = math.ceil(len(order)*q)
        head = set(order[:k])
        all_coverage = sum(counter[p] for p in head)/total_events
        candidate_coverage = candidate_df['pair'].isin(head).mean()
        rows.append({'top_pair_pct': 100*q, 'num_pairs': k,
                     'all_rejection_coverage_pct': 100*all_coverage,
                     'candidate_coverage_pct': 100*candidate_coverage,
                     'difference_pp': 100*(candidate_coverage-all_coverage)})
    return pd.DataFrame(rows)

fixed_tables = {'Overall': fixed_head_coverage(overall_frequency, df)}
for task in TASKS:
    fixed_tables[task] = fixed_head_coverage(task_frequency[task], df[df['task'] == task])
for name, table in fixed_tables.items():
    print(name); display(table.round(3))

## 1. 所有拒绝事件的长尾
这里不使用 Judge，仅使用完整 trace。横轴是按出现频率降序排列的不同 pair，纵轴是累计覆盖的拒绝事件比例。

In [ ]:
def tail_summary(counter):
    counts = np.array(sorted(counter.values(), reverse=True))
    n, total = len(counts), counts.sum()
    result = {'events': total, 'unique_pairs': n, 'singleton_pair_pct': 100*(counts==1).mean()}
    for q in [0.001, 0.01, 0.05, 0.10, 0.20]:
        k = max(1, math.ceil(n*q))
        result[f'top_{100*q:g}%_event_coverage'] = 100*counts[:k].sum()/total
    return result

tail_table = pd.DataFrame({k: tail_summary(v) for k, v in {**task_frequency, 'Overall': overall_frequency}.items()}).T
display(tail_table.round(2))
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for name, counter in {**task_frequency, 'Overall': overall_frequency}.items():
    counts = np.array(sorted(counter.values(), reverse=True))
    x = np.arange(1, len(counts)+1)/len(counts)*100
    y = np.cumsum(counts)/counts.sum()*100
    ax.plot(x, y, label=name, linewidth=2.3 if name == 'Overall' else 1.6)
ax.set(xlabel='Top pairs (%) ranked by trace frequency', ylabel='Cumulative rejection events (%)', xlim=(0, 100), ylim=(0, 100))
ax.legend(); ax.set_title('Long-tail distribution of all rejection pairs'); plt.show()

## 2. 完整 trace 中最高频的 10 个 pair
误拒候选率仅来自落入该 pair 的 3% Judge 样本，因此必须和 `judged_n` 一起解读。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True, trust_remote_code=True)
top_rows = []
for rank, (pair, count) in enumerate(overall_frequency.most_common(10), 1):
    sample = df[df['pair'] == pair]
    top_rows.append({
        'rank': rank, 'draft_id': pair[0], 'draft_text': repr(tokenizer.decode([pair[0]], clean_up_tokenization_spaces=False)),
        'residual_id': pair[1], 'residual_text': repr(tokenizer.decode([pair[1]], clean_up_tokenization_spaces=False)),
        'trace_frequency': count, 'trace_event_share_pct': 100*count/sum(overall_frequency.values()),
        **{task: task_frequency[task][pair] for task in TASKS},
        'judged_n': len(sample), 'supported_candidate_n': int(sample['supported_candidate'].sum()),
        'supported_candidate_rate': sample['supported_candidate'].mean() if len(sample) else np.nan,
    })
display(pd.DataFrame(top_rows).round(4))

## 3. 分箱工具与判读口径

每个分箱同时报告：样本数、`supported_candidate_rate`、`insufficient_rate`、`proven_invalid_rate`。只有第一项是当前有正面证据支持的误拒候选率。

In [ ]:
def quantile_relation(data, column, q=10, log_before_bin=False):
    work = data[[column, 'supported_candidate', 'insufficient', 'proven_invalid']].replace([np.inf, -np.inf], np.nan).dropna().copy()
    key = np.log1p(work[column]) if log_before_bin else work[column]
    work['bin'] = pd.qcut(key, q=q, duplicates='drop')
    out = work.groupby('bin', observed=True).agg(
        n=(column, 'size'), value_min=(column, 'min'), value_median=(column, 'median'), value_max=(column, 'max'),
        supported_candidate_rate=('supported_candidate', 'mean'), insufficient_rate=('insufficient', 'mean'),
        proven_invalid_rate=('proven_invalid', 'mean')).reset_index(drop=True)
    return out

def plot_relations(data, specs, title):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
    tables = {}
    for ax, (column, label, logbin) in zip(axes.flat, specs):
        table = quantile_relation(data, column, q=10, log_before_bin=logbin)
        tables[column] = table
        x = table['value_median']
        ax.plot(x, 100*table['supported_candidate_rate'], marker='o', label='supported candidate')
        ax.plot(x, 100*table['insufficient_rate'], marker='s', label='insufficient', alpha=.75)
        ax.plot(x, 100*table['proven_invalid_rate'], marker='^', label='proven invalid', alpha=.75)
        if logbin: ax.set_xscale('symlog', linthresh=1e-6)
        ax.set(xlabel=label, ylabel='Judge label rate (%)', title=f'{label} (equal-count bins)')
    axes.flat[0].legend(fontsize=9)
    fig.suptitle(title, fontsize=15); fig.tight_layout(); plt.show()
    return tables

## 4. Frequency 与误拒候选的关系

分别检查完整 trace frequency 与 calibration frequency。若 frequency 有判别力，`supported_candidate_rate` 应随频率呈稳定单调变化。

In [ ]:
frequency_specs = [
    ('trace_frequency_overall', 'Overall trace frequency', True),
    ('trace_frequency_task', 'Task-local trace frequency', True),
    ('calibration_frequency', 'Calibration frequency', True),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
frequency_tables = {}
for ax, (column, label, logbin) in zip(axes, frequency_specs):
    table = quantile_relation(df, column, q=10, log_before_bin=logbin); frequency_tables[column] = table
    ax.plot(table['value_median'], 100*table['supported_candidate_rate'], marker='o')
    ax.set_xscale('symlog', linthresh=1); ax.set(xlabel=label, ylabel='Supported candidate rate (%)', title=label)
fig.suptitle('Does pair frequency predict a supported mis-rejection candidate?'); fig.tight_layout(); plt.show()
for name, table in frequency_tables.items():
    print('\n', name); display(table.round(4))

## 4.1 每 1% pair 的潜在误拒候选率

按完整 trace frequency 对全部不同 pair 排名，再切成100个等pair数量区间。注意大量尾部pair的frequency都等于1，这些并列项内部的1%顺序是任意的；必须同时查看样本数和Wilson 95%置信区间。

In [ ]:
def wilson_interval(k, n, z=1.96):
    if n == 0: return np.nan, np.nan
    p = k/n; den = 1+z*z/n
    center = (p+z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-half, center+half

order = sorted(overall_frequency, key=overall_frequency.get, reverse=True)
rank = {pair: i for i, pair in enumerate(order)}
percentile_rows = []
for b in range(100):
    part = df[df['pair'].map(lambda p: min(99, int(rank[p]*100/len(order)))) == b]
    k = int(part['supported_candidate'].sum()); lo, hi = wilson_interval(k, len(part))
    start, end = math.floor(b*len(order)/100), math.floor((b+1)*len(order)/100)
    pairs = order[start:end]
    percentile_rows.append({
        'percentile': b+1, 'judged_n': len(part), 'candidate_rate_pct': 100*k/len(part) if len(part) else np.nan,
        'ci95_low_pct': 100*lo, 'ci95_high_pct': 100*hi,
        'frequency_median': np.median([overall_frequency[p] for p in pairs]),
        'frequency_min': min(overall_frequency[p] for p in pairs),
        'frequency_max': max(overall_frequency[p] for p in pairs),
    })
percentile_df = pd.DataFrame(percentile_rows)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(percentile_df['percentile'], percentile_df['candidate_rate_pct'], linewidth=1.4)
ax.fill_between(percentile_df['percentile'], percentile_df['ci95_low_pct'], percentile_df['ci95_high_pct'], alpha=.2)
ax.axhline(100*df['supported_candidate'].mean(), color='black', linestyle='--', label='overall rate')
ax.set(xlabel='Pair-frequency percentile (1 = highest)', ylabel='Supported candidate rate (%)', title='Candidate rate in every 1% pair-frequency bin')
ax.legend(); plt.show()
display(percentile_df.round(3))

## 5. Confidence gate 信号与误拒候选的关系

重点检查 draft/residual probability ratio；同时检查 target entropy、top-1/top-2 margin、draft probability、target top-1 probability 和 effective support size。

In [ ]:
confidence_specs = [
    ('draft_residual_probability_ratio', 'P(draft) / P(residual)', True),
    ('entropy_raw', 'Target entropy', False),
    ('target_top1_top2_margin', 'Top-1 minus top-2 probability', False),
    ('draft_target_probability', 'P_target(draft)', True),
    ('target_top1_probability', 'Target top-1 probability', False),
    ('effective_support_size', 'exp(entropy)', True),
]
confidence_tables = plot_relations(df, confidence_specs, 'Context-dependent confidence signals vs Judge labels')
for name, table in confidence_tables.items():
    print('\n', name); display(table.round(4))

## 6. 各数据集分别检查 ratio 与 entropy
用于识别整体趋势是否由某一个数据集驱动。

In [ ]:
fig, axes = plt.subplots(len(TASKS), 2, figsize=(12, 11), sharey=True)
for row, task in enumerate(TASKS):
    part = df[df['task'] == task]
    for col, (feature, label, logbin) in enumerate([
        ('draft_residual_probability_ratio', 'Probability ratio', True),
        ('entropy_raw', 'Target entropy', False),
    ]):
        table = quantile_relation(part, feature, q=8, log_before_bin=logbin)
        axes[row, col].plot(table['value_median'], 100*table['supported_candidate_rate'], marker='o')
        if logbin: axes[row, col].set_xscale('log')
        axes[row, col].set(title=f'{task}: {label}', xlabel=label, ylabel='Supported candidate rate (%)')
fig.tight_layout(); plt.show()

## 7. Gate 组合的观察性统计

这是已有事件上的观察性统计，不是因果实验。`would_force_accept` 与标签的关系只能用于诊断当前 gate 选择了什么事件。

In [ ]:
gate_table = df.groupby(['table_hit', 'probability_ratio_gate_passed', 'entropy_gate_passed', 'would_force_accept'], dropna=False).agg(
    n=('label', 'size'), supported_candidate_rate=('supported_candidate', 'mean'),
    insufficient_rate=('insufficient', 'mean'), proven_invalid_rate=('proven_invalid', 'mean'),
).reset_index().sort_values('n', ascending=False)
display(gate_table.assign(**{c: 100*gate_table[c] for c in ['supported_candidate_rate','insufficient_rate','proven_invalid_rate']}).round(3))

## 判读原则

1. 长尾由完整 trace 直接证明，不依赖 Judge。
2. 某特征具有判别力，应表现为误拒候选率随分箱出现稳定、跨数据集一致的单调趋势。
3. `INSUFFICIENT` 是缺失标签，不应直接合并为误拒；同时报告它是为了检查某些分箱是否更难判断。
4. 当前分析是相关性分析；真正选择 gate 阈值还需用精度、接受长度与吞吐端到端验证。